# IAPH - TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

## Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

### 1)&nbsp;Instalo las librerías necesarias para trabajar.

In [16]:
# Instalo las librerías necesarias para trabajar.
!pip install spacy --quiet
!python -m spacy download es_core_news_md --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 29.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


### 2)&nbsp;Importo las librerías necesarias para trabajar.

In [17]:
# Importo las librerías necesarias para trabajar.
import ipywidgets as widgets
import pandas as pd
import random
import spacy

from google.colab import data_table
from IPython.display import HTML, display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Actividades

### 1)&nbsp;Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. (3 puntos)

#### 1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

> Aplicación elegida: **Chatbot de consulta para estudiantes de secundaria**.
>
> El chatbot responderá preguntas sobre contenidos generales de distintas áreas, como ciencias naturales, matemática, historia, geografía y literatura, utilizando una base de preguntas y respuestas en español.
>
> Uso un dataset que encontré en Hugging Face: https://huggingface.co/datasets/Dontallo/QASpanis

#### 1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

In [18]:
# Defino la URL del dataset.
url_dataset = "https://huggingface.co/datasets/Dontallo/QASpanish/resolve/main/dataset.csv"

# Cargo el dataset en un DataFrame.
dataframe = pd.read_csv(url_dataset)

# Muestro el dataset.
dataframe

,pregunta,respuesta
0,¿Qué es el ciclo del agua?,"El ciclo del agua, también conocido como ciclo..."
1,¿Cómo se forman las montañas?,Las montañas se forman principalmente por la t...
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
3,¿Qué son los ecosistemas?,Un ecosistema es una comunidad de organismos v...
4,¿Qué es el sistema solar?,El sistema solar es un conjunto de objetos cel...
...,...,...
99,¿Cuál fue el primer presidente de Bolivia?,El primer presidente constitucional de Bolivia...
100,¿Cuántos lados tiene un cuadrado?,Un cuadrado es una figura geométrica plana que...
101,¿Qué son los productores en una cadena aliment...,"En una cadena alimentaria, los productores son..."
102,¿Qué evento desencadenó la Primera Guerra Mund...,El evento que desencadenó la Primera Guerra Mu...


In [19]:
# ¿Tengo preguntas repetidas? Busco por la palabra fotosíntesis.
preguntas_fotosintesis = dataframe[
    dataframe["pregunta"].str.contains("fotosíntesis", case=False, na=False)
]

# ¿Tengo preguntas repetidas? Muestro las preguntas encontradas.
preguntas_fotosintesis

,pregunta,respuesta
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
34,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
43,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
49,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso mediante el cual...
57,Descríbeme la fotosíntesis.,La fotosíntesis es el proceso mediante el cual...
88,Explícame la fotosíntesis.,La fotosíntesis es el proceso mediante el cual...


In [20]:
# Elimino las preguntas repetidas y conservo la primera aparición.
dataframe_limpio = dataframe.drop_duplicates(subset="pregunta", keep="first").reset_index(drop=True)

# Muestro el DataFrame actualizado.
dataframe_limpio

,pregunta,respuesta
0,¿Qué es el ciclo del agua?,"El ciclo del agua, también conocido como ciclo..."
1,¿Cómo se forman las montañas?,Las montañas se forman principalmente por la t...
2,¿Qué es la fotosíntesis?,La fotosíntesis es el proceso por el cual las ...
3,¿Qué son los ecosistemas?,Un ecosistema es una comunidad de organismos v...
4,¿Qué es el sistema solar?,El sistema solar es un conjunto de objetos cel...
...,...,...
91,¿Cuál fue el primer presidente de Bolivia?,El primer presidente constitucional de Bolivia...
92,¿Cuántos lados tiene un cuadrado?,Un cuadrado es una figura geométrica plana que...
93,¿Qué son los productores en una cadena aliment...,"En una cadena alimentaria, los productores son..."
94,¿Qué evento desencadenó la Primera Guerra Mund...,El evento que desencadenó la Primera Guerra Mu...


###  2)&nbsp;Crear el chatbot utilizando TF-IDF y similitud del coseno. (1 punto)

#### 2.1 Vectorizo las preguntas utilizando TF-IDF.

In [21]:
# Separo las preguntas y las respuestas del dataset.
preguntas = dataframe_limpio["pregunta"].tolist()
respuestas = dataframe_limpio["respuesta"].tolist()

# Vectorizo las preguntas utilizando TF-IDF.
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(preguntas)

# Muestro la matriz generada.
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 481 stored elements and shape (96, 140)>

> **tfidf_matrix** guarda la representación numérica de todas las preguntas para poder compararlas con una consulta nueva.

#### 2.2 Defino una función para encontrar la respuesta con mayor similitud.

In [22]:
# Defino una función para encontrar la respuesta con mayor similitud.
def responder_pregunta(consulta_usuario):
    # Convierto la consulta del usuario en un vector TF-IDF.
    consulta_vec = vectorizer.transform([consulta_usuario])

    # Comparo la consulta con todas las preguntas del dataset.
    similitudes = cosine_similarity(consulta_vec, tfidf_matrix).flatten()

    # Obtengo el índice de la pregunta con mayor similitud.
    indice_mejor_coincidencia = similitudes.argmax()

    # Devuelvo la respuesta correspondiente.
    return respuestas[indice_mejor_coincidencia]

> **argmax()** devuelve la posición del valor más alto de la lista de similitudes, es decir, identifica cuál es la pregunta del dataset que más se parece a la consulta del usuario.

#### 2.3 Pruebo una pregunta/respuesta rápida.

In [23]:
# Defino una consulta de prueba.
consulta_usuario = "¿Qué son los ecosistemas?"

# Obtengo la respuesta del chatbot.
respuesta = responder_pregunta(consulta_usuario)

# Muestro la respuesta obtenida.
print(respuesta)

Un ecosistema es una comunidad de organismos vivos que interactúan entre sí y con su entorno físico en un área específica. Los ecosistemas incluyen componentes bióticos (organismos vivos, como plantas, animales y microorganismos) y abióticos (factores no vivos, como el clima, el suelo, el agua y los nutrientes). Los ecosistemas pueden variar en tamaño, desde una pequeña charca hasta un gran bosque o un océano. Las interacciones dentro de un ecosistema incluyen procesos como la depredación, la competencia, la simbiosis y el ciclo de nutrientes. Los ecosistemas son dinámicos y pueden cambiar con el tiempo debido a factores naturales y humanos.


#### 2.4 Chatbot TF-IDF.

In [24]:
# Creo la caja de texto para escribir las preguntas.
entrada = widgets.Text(
    description="Pregunta:",
    layout=widgets.Layout(width="90%")
)

# Selecciono preguntas aleatorias del dataset como sugerencias.
preguntas_sugeridas = dataframe_limpio["pregunta"].sample(5).tolist()

# Construyo el texto con las preguntas de ejemplo en una sola línea.
texto_ejemplos = ", ".join(f'"{pregunta}"' for pregunta in preguntas_sugeridas)

# Creo una leyenda con preguntas de ejemplo.
sugerencias = HTML(
    f'<div style="width: 90%; color: #777; font-size: 12px; margin-top: 5px; padding-left: 89px;">'
    f'Ej: {texto_ejemplos}.'
    f'</div>'
)

# Defino la interacción con el chatbot.
def responder_chat(widget):
    consulta_usuario = widget.value

    # Obtengo la respuesta del chatbot.
    respuesta = responder_pregunta(consulta_usuario)

    # Muestro la pregunta y su respuesta.
    display(HTML(
        f'<div style="width: 90%; color: #333; font-size: 13px; margin-top: 15px; padding-left: 0;">'
        f'<b>Pregunta:</b> {consulta_usuario}<br>'
        f'<b>Respuesta:</b> {respuesta}'
        f'</div>'
    ))

    # Limpio la caja de texto.
    widget.value = ""

# Envío la pregunta al presionar Enter.
entrada.on_submit(responder_chat)

# Muestro la caja de texto y las sugerencias.
display(entrada, sugerencias)

Text(value='', description='Pregunta:', layout=Layout(width='90%'))

### 3)&nbsp;Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado de las librerías Spacy o nltk o gensim eligió. Justifique.

> Embedding elegido: **modelo preentrenado `es_core_news_md` de spaCy**.
>
> Elijo este modelo porque trabaja con texto en español y me permite obtener representaciones vectoriales de las preguntas para comparar su similitud semántica. Considero que resulta adecuado para el dataset que utilizo, ya que contiene preguntas y respuestas en español.

#### 3.1 Vectorizo las preguntas utilizando embeddings.

In [25]:
# Cargo el modelo preentrenado en español.
modelo_embeddings = spacy.load("es_core_news_md")

# Obtengo los vectores de todas las preguntas del dataset.
vectores_preguntas = [
    modelo_embeddings(pregunta).vector
    for pregunta in preguntas
]

# Muestro la cantidad de vectores generados.
len(vectores_preguntas)

96

#### 3.2 Defino una función para encontrar la respuesta con mayor similitud.

In [26]:
# Defino una función para encontrar la respuesta con mayor similitud semántica.
def responder_pregunta_embeddings(consulta_usuario):
    # Obtengo el vector de la consulta del usuario.
    vector_consulta = modelo_embeddings(consulta_usuario).vector

    # Comparo la consulta con todos los vectores de las preguntas.
    similitudes = cosine_similarity(
        [vector_consulta],
        vectores_preguntas
    ).flatten()

    # Obtengo el índice de la pregunta con mayor similitud.
    indice_mejor_coincidencia = similitudes.argmax()

    # Devuelvo la respuesta correspondiente.
    return respuestas[indice_mejor_coincidencia]

#### 3.3 Chatbot embeddings.

In [27]:
# Creo la caja de texto para escribir las preguntas.
entrada = widgets.Text(
    description="Pregunta:",
    layout=widgets.Layout(width="90%")
)

# Selecciono preguntas aleatorias del dataset como sugerencias.
preguntas_sugeridas = dataframe_limpio["pregunta"].sample(5).tolist()

# Construyo el texto con las preguntas de ejemplo en una sola línea.
texto_ejemplos = ", ".join(f'"{pregunta}"' for pregunta in preguntas_sugeridas)

# Creo una leyenda con preguntas de ejemplo.
sugerencias = HTML(
    f'<div style="width: 90%; color: #777; font-size: 12px; margin-top: 5px; padding-left: 89px;">'
    f'Ej: {texto_ejemplos}.'
    f'</div>'
)

# Defino la interacción con el chatbot.
def responder_chat_embeddings(widget):
    consulta_usuario = widget.value

    # Obtengo la respuesta del chatbot.
    respuesta = responder_pregunta_embeddings(consulta_usuario)

    # Muestro la pregunta y su respuesta.
    display(HTML(
        f'<div style="width: 90%; color: #333; font-size: 13px; margin-top: 15px; padding-left: 0;">'
        f'<b>Pregunta:</b> {consulta_usuario}<br>'
        f'<b>Respuesta:</b> {respuesta}'
        f'</div>'
    ))

    # Limpio la caja de texto.
    widget.value = ""

# Envío la pregunta al presionar Enter.
entrada.on_submit(responder_chat_embeddings)

# Muestro la caja de texto y las sugerencias.
display(entrada, sugerencias)

Text(value='', description='Pregunta:', layout=Layout(width='90%'))

### 4)&nbsp;Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

#### 4.1 Tomo algunas preguntas al azar y las modifico para que no sean exactas.

In [28]:
# Defino una función para modificar las preguntas originales.
def modificar_pregunta(pregunta):
    modificaciones = {
        "¿Qué es ": ["Explícame qué es ", "¿En qué consiste ", "¿Cómo definirías "],
        "¿Qué son ": ["Explícame qué son ", "¿Cómo definirías ", "¿A qué se llama "],
        "¿Cuál es ": ["Decime cuál es ", "Quiero saber cuál es ", "¿Podés decirme cuál es "],
        "¿Cuáles son ": ["Decime cuáles son ", "Quiero saber cuáles son ", "¿Podés decirme cuáles son "],
        "¿Cómo ": ["¿De qué manera ", "Explícame cómo ", "Quiero saber cómo "],
        "¿Por qué ": ["Explícame por qué ", "Quiero saber por qué ", "¿Cuál es la razón por la que "],
        "¿Quién fue ": ["Contame quién fue ", "Quiero saber quién fue ", "¿Podés explicarme quién fue "]
    }

    # Busco una forma alternativa de formular la pregunta.
    for inicio, alternativas in modificaciones.items():
        if pregunta.startswith(inicio):
            return random.choice(alternativas) + pregunta[len(inicio):]

    # Si no encuentro un patrón, modifico la forma general de la pregunta.
    return "Quiero saber: " + pregunta.replace("¿", "").replace("?", "")


# Selecciono quince preguntas aleatorias del dataset.
muestra = dataframe_limpio.sample(15).copy()

# Convierto el índice original en una columna identificadora.
muestra = muestra.reset_index().rename(
    columns={
        "index": "id",
        "pregunta": "pregunta_original"
    }
)

# Genero una versión modificada de cada pregunta.
muestra["pregunta_modificada"] = muestra["pregunta_original"].apply(modificar_pregunta)

# Muestro las preguntas seleccionadas.
muestra[["id", "pregunta_original", "pregunta_modificada"]]

,id,pregunta_original,pregunta_modificada
0,41,¿Qué es un microscopio?,Explícame qué es un microscopio?
1,74,Explícame cómo se forman los volcanes.,Quiero saber: Explícame cómo se forman los vol...
2,88,Explícame la teoría atómica.,Quiero saber: Explícame la teoría atómica.
3,52,Descríbeme el proceso de digestión en los sere...,Quiero saber: Descríbeme el proceso de digesti...
4,35,¿Qué es la traslación de la Tierra?,¿Cómo definirías la traslación de la Tierra?
5,13,¿Qué es la atmósfera?,¿Cómo definirías la atmósfera?
6,47,¿Qué es la energía eólica?,¿En qué consiste la energía eólica?
7,92,¿Cuántos lados tiene un cuadrado?,Quiero saber: Cuántos lados tiene un cuadrado
8,58,Descríbeme el proceso de respiración celular.,Quiero saber: Descríbeme el proceso de respira...
9,60,Descríbeme el sistema inmunológico.,Quiero saber: Descríbeme el sistema inmunológico.


#### 4.2 Le pregunto a los dos tipos de chatbots y comparo los resultados.

In [29]:
# Obtengo las respuestas de ambos chatbots para cada pregunta modificada.
muestra["respuesta_tfidf"] = muestra["pregunta_modificada"].apply(responder_pregunta)
muestra["respuesta_embeddings"] = muestra["pregunta_modificada"].apply(responder_pregunta_embeddings)

# Indico si cada chatbot devolvió exactamente la respuesta esperada.
muestra["acierto_tfidf"] = muestra["respuesta_tfidf"] == muestra["respuesta"]
muestra["acierto_embeddings"] = muestra["respuesta_embeddings"] == muestra["respuesta"]

# Creo una tabla con los resultados.
tabla_resultados = muestra[[
    "id",
    "pregunta_original",
    "pregunta_modificada",
    "respuesta",
    "respuesta_tfidf",
    "acierto_tfidf",
    "respuesta_embeddings",
    "acierto_embeddings"
]].copy()

# Agrego una fila final con el total de aciertos.
tabla_resultados.loc["TOTAL ACIERTOS"] = [
    "",
    "",
    "",
    "",
    "",
    tabla_resultados["acierto_tfidf"].sum(),
    "",
    tabla_resultados["acierto_embeddings"].sum()
]

# Muestro la tabla comparativa.
tabla_resultados

,id,pregunta_original,pregunta_modificada,respuesta,respuesta_tfidf,acierto_tfidf,respuesta_embeddings,acierto_embeddings
0,41,¿Qué es un microscopio?,Explícame qué es un microscopio?,Un microscopio es un instrumento que permite o...,Un microscopio es un instrumento que permite o...,1,Un microscopio es un instrumento que permite o...,1
1,74,Explícame cómo se forman los volcanes.,Quiero saber: Explícame cómo se forman los vol...,Los volcanes se forman cuando el magma del int...,Los volcanes se forman cuando el magma del int...,1,Los volcanes se forman cuando el magma del int...,1
2,88,Explícame la teoría atómica.,Quiero saber: Explícame la teoría atómica.,La teoría atómica es la idea científica de que...,La teoría atómica es la idea científica de que...,1,La teoría atómica es la idea científica de que...,1
3,52,Descríbeme el proceso de digestión en los sere...,Quiero saber: Descríbeme el proceso de digesti...,La digestión es el proceso mediante el cual lo...,La digestión es el proceso mediante el cual lo...,1,La digestión es el proceso mediante el cual lo...,1
4,35,¿Qué es la traslación de la Tierra?,¿Cómo definirías la traslación de la Tierra?,La traslación de la Tierra es el movimiento de...,La traslación de la Tierra es el movimiento de...,1,La rotación de la Tierra es el giro de nuestro...,0
5,13,¿Qué es la atmósfera?,¿Cómo definirías la atmósfera?,La atmósfera es la capa de gases que rodea la ...,La atmósfera es la capa de gases que rodea la ...,1,La atmósfera es la capa de gases que rodea la ...,1
6,47,¿Qué es la energía eólica?,¿En qué consiste la energía eólica?,La energía eólica es la energía obtenida del v...,La energía eólica es la energía obtenida del v...,1,La energía eólica es la energía obtenida del v...,1
7,92,¿Cuántos lados tiene un cuadrado?,Quiero saber: Cuántos lados tiene un cuadrado,Un cuadrado es una figura geométrica plana que...,Un cuadrado es una figura geométrica plana que...,1,Un cuadrado es una figura geométrica plana que...,1
8,58,Descríbeme el proceso de respiración celular.,Quiero saber: Descríbeme el proceso de respira...,La respiración celular es el proceso mediante ...,La respiración celular es el proceso mediante ...,1,La respiración celular es el proceso mediante ...,1
9,60,Descríbeme el sistema inmunológico.,Quiero saber: Descríbeme el sistema inmunológico.,El sistema inmunológico es la defensa del cuer...,El sistema inmunológico es la defensa del cuer...,1,El sistema digestivo es responsable de descomp...,0


#### 4.3 Tabla interactiva para ver preguntas y respuestas, para entender los resultados.

In [30]:
# Creo una copia del DataFrame limpio.
tabla_dataset = dataframe_limpio.copy()

# Creo los filtros.
busqueda_pregunta = widgets.Text(
    placeholder="Buscar en preguntas...",
    layout=widgets.Layout(width="100%")
)

busqueda_respuesta = widgets.Text(
    placeholder="Buscar en respuestas...",
    layout=widgets.Layout(width="100%")
)

# Creo los botones para limpiar los filtros.
limpiar_pregunta = widgets.Button(
    description="×",
    layout=widgets.Layout(width="35px")
)

limpiar_respuesta = widgets.Button(
    description="×",
    layout=widgets.Layout(width="35px")
)

# Creo el espacio para mostrar la tabla.
salida_busqueda = widgets.Output()

# Defino la búsqueda.
def buscar_dataset(change=None):
    resultados = tabla_dataset.copy()

    if busqueda_pregunta.value:
        resultados = resultados[
            resultados["pregunta"].str.contains(
                busqueda_pregunta.value,
                case=False,
                na=False
            )
        ]

    if busqueda_respuesta.value:
        resultados = resultados[
            resultados["respuesta"].str.contains(
                busqueda_respuesta.value,
                case=False,
                na=False
            )
        ]

    with salida_busqueda:
        salida_busqueda.clear_output(wait=True)

        display(
            resultados.head(10).style.set_table_attributes(
                'style="width:100%; table-layout:fixed;"'
            )
        )

# Actualizo la tabla cuando cambia un filtro.
busqueda_pregunta.observe(buscar_dataset, names="value")
busqueda_respuesta.observe(buscar_dataset, names="value")

# Limpio cada filtro al presionar la cruz.
limpiar_pregunta.on_click(lambda _: setattr(busqueda_pregunta, "value", ""))
limpiar_respuesta.on_click(lambda _: setattr(busqueda_respuesta, "value", ""))

# Muestro los filtros con espacio entre ambos grupos.
display(
    widgets.HBox(
        [
            widgets.HBox(
                [busqueda_pregunta, limpiar_pregunta],
                layout=widgets.Layout(width="49%")
            ),
            widgets.HBox(
                [busqueda_respuesta, limpiar_respuesta],
                layout=widgets.Layout(width="49%")
            )
        ],
        layout=widgets.Layout(
            width="100%",
            justify_content="space-between"
        )
    ),
    salida_busqueda
)

# Muestro inicialmente los primeros diez registros.
buscar_dataset()

Output()

### 5)&nbsp;Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.

> En esta prueba, TF-IDF obtuvo más aciertos que el modelo de embeddings. Considero que esto ocurre porque las preguntas modificadas conservan muchas veces el concepto principal de la pregunta original. En TF-IDF, esas palabras específicas tienen mucho peso y permiten diferenciar con facilidad una pregunta de otra.
>
> Por ejemplo, términos como:
>
> - **biodiversidad**
> - **continentes**
> - **clima**
> - **magnetismo**
>
> son muy útiles para identificar el tema correcto, aunque cambie la forma de hacer la pregunta.
>
> En cambio, el modelo de embeddings representa la frase completa como un vector. Esto puede hacer que dos preguntas con una estructura parecida queden demasiado próximas aunque el concepto principal sea distinto. Por ejemplo:
>
> - **“¿Qué son los continentes?”** y **“¿Qué es el magnetismo?”** tienen una estructura muy similar, pero las palabras importantes, **continentes** y **magnetismo**, pertenecen a temas completamente diferentes.
>
> También hubo errores menos explicables por la estructura, por ejemplo:
>
> - **“¿En qué consiste la biodiversidad?”** → devolvió una respuesta sobre la Primera Guerra Mundial.
> - **“¿En qué consiste el clima?”** → devolvió una respuesta sobre el corazón.
>
> Esto muestra que, en algunas consultas, el embedding no da suficiente peso al concepto central y termina considerando más cercana otra pregunta que no corresponde. En este dataset, como las preguntas son cortas y contienen términos muy específicos, TF-IDF funciona mejor porque esas palabras clave resultan muy discriminantes.

## No olvides

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes.

> En este trabajo comparé dos chatbots de recuperación: uno basado en **TF-IDF** y otro en **embeddings con `es_core_news_md` de spaCy**.
>
> Probé ambos chatbots de forma manual, mediante cajas de texto en las que se pueden escribir distintas consultas y observar las respuestas obtenidas.
>
> Además, realicé una prueba automática en la que seleccioné preguntas al azar del dataset, modifiqué su redacción y comparé las respuestas de ambos modelos.
>
> En las pruebas realizadas, **TF-IDF obtuvo mejores resultados**, especialmente porque muchas preguntas contienen términos específicos que permiten identificar bien el tema. El modelo de embeddings presentó más confusiones entre preguntas distintas, lo que muestra que un método más complejo no necesariamente funciona mejor y que el resultado también depende de las características del dataset.
>
> **Fuentes utilizadas:**
> - Notebooks de clase.
> - Búsquedas en Internet.
> - Dataset de Hugging Face.
> - Documentación de spaCy y scikit-learn.
> - ChatGPT y Gemini para esclarecer dudas.